# Neovariants Master Table 

### Pending 
- Clean code
- UMI combination analysis

In [140]:
#Load packages
library(tidyverse)
library(scales)
suppressMessages(library(BSgenome))
library(ggplot2)
library(dplyr)
library(broom) 
library(RColorBrewer)
library(DT)
library("stringi")
library(data.table)
library(ggalluvial)

## 1. Tidy data

In [141]:
df_summary <- read.csv("output/df_summary_complete.csv")

In [142]:
#data conversion define hierarchy
df_summary_tidy <- df_summary %>% 
separate(variation, into = c("variant1", "variant2"), sep = "-",remove = FALSE) %>% 
separate(umis, into = c("umis1", "umis2"), sep = "-",remove = FALSE) %>%
  mutate(original = case_when(
    nucl_po == variant1 ~ variant1,
    nucl_po == variant2 ~ variant2,
    TRUE ~ "unknown"
  )) %>%
  mutate(neovariant = case_when(
    nucl_po != variant1 & nucl_po == variant2 ~ variant1,
    nucl_po != variant2 & nucl_po == variant1 ~ variant2,
    nucl_po != variant1 & nucl_po != variant2 ~ "unknown",
    TRUE ~ NA
  )) %>%
  mutate(neovariant = case_when(
    nucl_po != variant1 & nucl_po == variant2 ~ variant1,
    nucl_po != variant2 & nucl_po == variant1 ~ variant2,
    nucl_po != variant1 & nucl_po != variant2 ~ "unknown",
    TRUE ~ NA
  )) %>%


  mutate(original2 = case_when(
    nucl_po == variant1 ~ variant1,
    nucl_po == variant2 ~ variant2,
    TRUE ~ NA
  )) %>%
  mutate(neovariant1 = case_when(
    nucl_po != variant1 & nucl_po == variant2 ~ variant1,
    nucl_po != variant2 & nucl_po == variant1 ~ variant2,
    nucl_po != variant1 & nucl_po != variant2 ~ variant1,
    TRUE ~ NA
  )) %>%
  mutate(neovariant2 = case_when(
    original == "unknown" & neovariant == "unknown" ~ variant2,
  )) %>% 
  mutate(original_umis = case_when(
    nucl_po == variant1 ~ umis1,
    nucl_po == variant2 ~ umis2,
    TRUE ~ NA
  )) %>%
  mutate(neovariant1_umis = case_when(
    nucl_po != variant1 & nucl_po == variant2 ~ umis1,
    nucl_po != variant2 & nucl_po == variant1 ~ umis2,
    nucl_po != variant1 & nucl_po != variant2 ~ umis1,
    TRUE ~ NA
  )) %>%
  mutate(neovariant2_umis = case_when(
    original == "unknown" & neovariant == "unknown" ~ umis2,
    TRUE ~ NA
      )) %>% select(-aid_motif1,-aid_motif2,-X)

In [143]:
# read data from Kees and change vgene_position_aligned `9999´ to allow posterior left_join
#events_v2 <- readxl::read_xlsx("~/repositories/FL_10X_2/310_Caught_in_the_act_AID/outs/paperTables/events.V2.0.xlsx") %>% 
events_v2 <- readxl::read_xlsx("input/events.V2.0.xlsx") %>% 
          mutate(vgene_position_aligned = case_when(vgene_position_aligned == 9999 ~ NA,
                                                   TRUE ~ vgene_position_aligned))

In [144]:
#Join Kees data and my data
events_v3 <- events_v2 %>% select(-nucl_po,-`AID motif`,-productive) %>% left_join(df_summary_tidy, by=c("subject","cell","subregion","variation","context_po","umis","vgene_position_aligned")) 

In [145]:
#reorder according to Kees order
events_v3 <- events_v3 %>% select(
 order,
  Chip,
  Sample,
  `Ig Chain`,
  `V(D)Jmutations`,
  `VDJmutations`,
  `VJmutations`,
  inBoth,
  `Cell ident`,
  subject,
  cell,
  subregion,
  position,
  vgene_position_aligned,
  context_po,
#  aid_motif,
  original2,
  neovariant1,
  neovariant2,
  original_umis,
  neovariant1_umis,
  neovariant2_umis,
  productive
) %>% dplyr::rename("original"="original2") 

In [146]:
WriteXLS::WriteXLS(events_v3,
                    "output/events.v3.0.xlsx" )


## 2. Calculate proportion FW/CDR in events

In [147]:
subregion <- df_summary %>% group_by(subregion) %>% count() %>% mutate(type = ifelse(str_detect(subregion, "^CDR"), "CDR", "FR"))
subregion

subregion,n,type
<chr>,<int>,<chr>
CDR1,152,CDR
CDR2,45,CDR
CDR3,91,CDR
FR1,284,FR
FR2,324,FR
FR3,302,FR
FR4,41,FR


In [148]:
subregion %>% group_by(type) %>% summarize(n_event=sum(n)) %>% mutate(Percentage =n_event * 100 / sum(n_event))

type,n_event,Percentage
<chr>,<int>,<dbl>
CDR,288,23.24455
FR,951,76.75545


## 3. Calculate the timing of neovariant

In [149]:
events_v3 <- readxl::read_xlsx("output/events.v3.0.xlsx")

In [150]:
events_v31 <- events_v3 %>% 
             mutate(h_from_event = log(original_umis/(original_umis+neovariant1_umis))/-0.231049) 


#### Cell count with neovariants by case

In [151]:
shm_cell_patient <- events_v31 %>% 
   group_by(Sample) %>%
   summarise(neovariant_cells = n_distinct(cell))

shm_cell_patient

Sample,neovariant_cells
<chr>,<int>
S10000,128
S11770,1
S12500,1
S13530,87
S13553,103
S144,40
S8934,4


In [152]:
# Total number of cells

print("Number of TOTAL cells with neovariants")
shm_cell_patient %>% pull(neovariant_cells) %>% sum()

[1] "Number of TOTAL cells with neovariants"


[1] 364

#### Events number by case

In [153]:
c_pos <-events_v31 %>%
  group_by(Sample) %>%
  summarise(n_events = n())

c_pos

Sample,n_events
<chr>,<int>
S10000,394
S11770,1
S12500,2
S13530,400
S13553,322
S144,116
S8934,4


#### Include a numeric cell ID (n_cell_ID)

In [154]:
# Extracting the ID numbers after "S" and creating a consecutive number according to the cellbarcode
events_v31 <- events_v31 %>%
  mutate(sample_number = as.numeric(sub("S", "", Sample))) %>%
  group_by(sample_number) %>%
  mutate(n_cell_ID = paste(sample_number, dense_rank(cell), sep = ".")) %>%
  ungroup() %>%
  select(-sample_number)

In [155]:
WriteXLS::WriteXLS(events_v31,
                    "output/events.v3.1.xlsx" )

## 4. Calculate cell numbers 

In [ ]:
df_seq <- read.csv("output/UMI_consensus_all.csv")
head(df_seq)

In [ ]:
case_cell <- df_seq %>% 
   mutate(Patient_id = str_extract(subject, "(?<=_)[^_]+(?=-)"),diagnosis = case_when(str_detect(subject, "^K") ~ "FL",
                         str_detect(subject, "^Q") ~ "CLL",
                          TRUE ~ NA)) %>%
  mutate(experiment =sub("_(.*)$", "", subject)) %>%
   group_by(diagnosis,Patient_id) %>% #experiment
    summarise(total_cell = n_distinct(cell)) %>%
    replace(is.na(.), 0) 
case_cell

### D) Number of event (positions) by cell

In [ ]:
#pos by cell
pbc <- df_summary %>% #filter(source == "K2B_S12500_L")
   group_by(subject, cell) %>%
   summarise(pos_by_cell = n_distinct(vgene_position_aligned)) %>%
   mutate(case = str_extract(subject, "(?<=_)[^_]+(?=-)"))
head(pbc)

In [ ]:
pbc %>% group_by(pos_by_cell) %>% summarize(n=n()) %>% mutate(percentage= n * 100 / sum(n))

## 5. Incorporate Syn/NonSyn data (Check problem of UMIS with NA)

In [ ]:
events_v31 <- readxl::read_xlsx("output/events.v3.1.xlsx")

In [ ]:
## Load all SHMss events detected (positive + false positive)
df <- Sys.glob("input/results_20231120/FL/*/*.csv") %>%
  purrr::map_dfr(function(x) {
    readr::read_csv(x, col_types = "ccicccilciicccccc", progress = FALSE) %>%
      dplyr::mutate(subject = gsub("^.*/|\\.shm.*$", "", x))
  })

In [ ]:
head(df)

In [ ]:
options(repr.matrix.max.cols=50, repr.matrix.max.rows=100)
events_v32 <- events_v31 %>% left_join(df %>% select(subject,cell,subregion,position,codon_cc,aa_ref_cc,aa_alt_cc,nucl,context_cc,nucl_po), by=c("subject","cell","subregion","position")) %>% unique() %>% 
mutate(aminoacid_original=if_else(nucl == original,aa_alt_cc,NA)) %>% 
mutate(aminoacid_n1=if_else(nucl == neovariant1,aa_alt_cc,NA)) %>%
mutate(aminoacid_n2=if_else(nucl == neovariant2,aa_alt_cc,NA)) %>%
 group_by(order,Chip,Sample,`Ig Chain`,	`V(D)Jmutations`,	`VDJmutations`,
          `VJmutations`,`inBoth`,`Cell ident`,subject,cell,subregion,position,vgene_position_aligned,
          context_po,context_cc,nucl_po,original,neovariant1,neovariant2,original_umis,neovariant1_umis,neovariant2_umis,productive,h_from_event,n_cell_ID,codon_cc) %>%
  summarise(
    aminoacid_original = na.omit(aminoacid_original)[1],
    aminoacid_n1 = na.omit(aminoacid_n1)[1],
    aminoacid_n2 = na.omit(aminoacid_n2)[1],
    .groups = 'drop' # to drop the grouping after summarising
  ) %>% mutate(
    mutation_type = case_when(
      aminoacid_original == aminoacid_n1 ~ "silent",
      (!is.na(aminoacid_original)) & aminoacid_n1 == "*" ~ "nonsense",
      aminoacid_original == "*" ~ "stop-loss",
      aminoacid_original != aminoacid_n1 ~ "missense",
      TRUE ~ NA_character_
    )
  )

### 6. Include modifications to master table like:
 - Generate full context (5 nt) 
 - Generate a full context (5 nt) for every neovariant 1 and 2 when there is not original
 - Calculate again the AID motif now using the cell context
- Define in which strand it was found the motif
 - Identify if the motif are destroyed with the neovariant

In [ ]:
#events_v33 <- readxl::read_xlsx("output/events.v3.1.xlsx")

In [ ]:
# Generate full context (5 nt) and also with context_cc
events_v33 <- events_v32 %>%
              mutate(nucl_po = toupper(nucl_po)) %>% mutate(context_po_coding_strand = stri_replace_first_fixed(context_po, ".", nucl_po)) %>% 
              mutate(context_cc_po = stri_replace_first_fixed(context_cc, ".", original)) %>% 
              mutate(variation_delete = case_when(is.na(original) ~ NA, !is.na(original) ~ paste0(original,"-",neovariant1))) %>%
              mutate(variation_delete2 = case_when(is.na(original) ~ paste0(neovariant1,"-",neovariant2), !is.na(original) ~ NA))

In [ ]:
head(events_v33)

### Identify AID motifs

In [ ]:
# Modification of AID motif to identify motif forward and reverse
aidp2 <- list(
  # Canonical AID signature should be C>T/G RCY
  WRCY = c(f = c(m = "C-[TGA]", c = "[AT][AG].[CT].")), #WRCY (forw) and RGYW (Rev)
  # non-canonical according to Kasar A>C at WA
  WA = c(f = c(m = "A-[TGC]", c = ".[AT].[ACGT].")),
  # signature 9 accordign to Alexandrov C>T at N.G
  RCG = c(f = c(m = "C-[TGA]", c = ".[AG].G."))
)

In [ ]:
# Function to get reverse complement, handles NA
get_rev_comp <- function(s) {
  if (is.na(s)) return(NA)
  dna_s <- DNAString(s)
  return(as.character(reverseComplement(dna_s)))
}

# Custom function to get complement without reversing
get_complement <- function(seq) {
  # Define the complement mapping
  complement_map <- c("A" = "T", "T" = "A", "G" = "C", "C" = "G", "-" = "-", "N" = "N")
  
  # Convert sequence string to character vector
  seq_vector <- strsplit(seq, '')[[1]]
  
  # Find the complement using the mapping
  complement_vector <- complement_map[seq_vector]
  
  # Convert back to a single string
  complement_seq <- paste0(complement_vector, collapse = "")
  
  return(complement_seq)
}

In [ ]:
identify_aid_patterns_bystrand <- function(mutation, context, aid_patterns = aidp2) {
      matches <- names(aid_patterns)[sapply(aid_patterns, function(p) {
        (grepl(p["f.m"], mutation) & grepl(p["f.c"], context))
      })]
    
      if (length(matches) == 1) {
        return(matches)
      }
      else if (length(matches) == 0) {
        return("None")
      }
      else {
        print("Problem!")
      }
    }

In [ ]:
#Define AID motifs from consensus cell context (only for variation where is possible to define hierarchy)
events_v33$original_aid_motif1 <- mapply(identify_aid_patterns_bystrand, events_v33$variation_delete, events_v33$context_cc)
events_v33$original_aid_motif2 <- mapply(identify_aid_patterns_bystrand, sapply(events_v33$variation_delete, get_complement), sapply(events_v33$context_cc, get_rev_comp))

events_v33$neovariant_aid_motif1 <- mapply(identify_aid_patterns_bystrand, sapply(events_v33$variation_delete, reverse), events_v33$context_cc)
events_v33$neovariant_aid_motif2 <- mapply(identify_aid_patterns_bystrand, sapply(sapply(events_v33$variation_delete, reverse),get_complement), sapply(events_v33$context_cc, get_rev_comp))

In [ ]:
#Define AID motifs from consensus cell context when there is not original nucleotide

# AID patterns; f/r = forward/reverse, m/c = mutation/context
aidp <- list(
  # Canonical AID signature should be C>T/G RCY
  WRCY = c(f = c(m = "C-[TGA]", c = "[AT][AG].[CT]."), r = c(m = "G-[ACT]", c = ".[AG].[CT][AT]")), #WRCY (forw) and RGYW (Rev)
  # non-canonical according to Kasar A>C at WA
  WA = c(f = c(m = "A-[TGC]", c = ".[AT].[ACGT]."), r = c(m = "T-[CGA]", c = ".[ACGT].[AT].")),
  # signature 9 accordign to Alexandrov C>T at N.G
  RCG = c(f = c(m = "C-[TGA]", c = ".[AG].G."), r = c(m = "G-[ACT]", c = ".C.[CT]."))
)

identify_aid_patterns <- function(mutation, context, aid_patterns = aidp) {
      matches <- names(aid_patterns)[sapply(aid_patterns, function(p) {
        (grepl(p["f.m"], mutation) & grepl(p["f.c"], context)) |
          (grepl(p["r.m"], mutation) & grepl(p["r.c"], context))
      })]
    
      if (length(matches) == 1) {
        return(matches)
      }
      else if (length(matches) == 0) {
        return("None")
      }
      else {
        print("Problem!")
      }
    }

reverse_string <- function(x) {
  if (is.na(x)) {
    return(NA)
  }
  elements <- strsplit(x, "-")[[1]]
  reversed_elements <- rev(elements)
  return(paste(reversed_elements, collapse = "-"))
}

# Calculate AID motif for context cc with neovariant 1 or neovariant 2
events_v33$neovariant_1_aid_motif <- mapply(identify_aid_patterns, events_v33$variation_delete2, events_v33$context_cc)
events_v33$neovariant_2_aid_motif <- mapply(identify_aid_patterns, sapply(events_v33$variation_delete2, reverse_string), events_v33$context_cc)

In [ ]:
# Rank motifs
events_v33 <- events_v33 %>% 
  mutate(original_aid_motif = case_when(is.na(original) ~ "Not_evaluated",
    original_aid_motif1 == "WRCY" | original_aid_motif2 == "WRCY" ~ "WRCY",
    original_aid_motif1 == "WA" | original_aid_motif2 == "WA" ~ "WA",
    original_aid_motif1 == "RCG" | original_aid_motif2 == "RCG" ~ "RCG",
    original_aid_motif1 == "None" | original_aid_motif2 == "None" ~ "None",
    TRUE ~ NA
  )) %>%  
    mutate(neovariant_aid_motif = case_when(
    is.na(original) ~ "Not_evaluated",
    neovariant_aid_motif1 == "WRCY" | neovariant_aid_motif2 == "WRCY" ~ "WRCY",
    neovariant_aid_motif1 == "WA" | neovariant_aid_motif2 == "WA" ~ "WA",
    neovariant_aid_motif1 == "RCG" | neovariant_aid_motif2 == "RCG" ~ "RCG",
    neovariant_aid_motif1 == "None" | neovariant_aid_motif2 == "None" ~ "None",
    TRUE ~ NA
  )) %>% 
  mutate(motif_destroyed = case_when(
    original_aid_motif == "None"  ~ "No_motif", #& neovariant_aid_motif == "None"
        is.na(variation_delete) ~ "No_evaluated",
    original_aid_motif == neovariant_aid_motif ~ "FALSE",
    !is.na(original_aid_motif) & is.na(neovariant_aid_motif) ~ "TRUE",
    original_aid_motif != neovariant_aid_motif ~ "TRUE",
    is.na(original_aid_motif) ~ "No_motif",
    TRUE ~ "Other" # catches all other cases, can be removed if not needed
  ))  %>%
  mutate(strand_of_motif = case_when(
    original_aid_motif1 != "None" ~ "coding",
    original_aid_motif2 != "None" ~ "non-coding",
    is.na(original) ~ "Not_evaluated",
    original_aid_motif1 == "None" & original_aid_motif2 == "None" ~ "No-motif",
    TRUE ~ "Other"  # For any other cases, can be removed if not needed
   )) %>%
  mutate(motif_destroyed_neovariant = case_when(
    neovariant_aid_motif == "None"  ~ "No_motif",
        is.na(original) ~ "No_evaluated",
    original_aid_motif == neovariant_aid_motif ~ "FALSE",
    !is.na(original_aid_motif) & is.na(neovariant_aid_motif) ~ "TRUE",
    original_aid_motif != neovariant_aid_motif ~ "TRUE",
    is.na(neovariant_aid_motif) ~ "No_motif",
    TRUE ~ "Other" # catches all other cases, can be removed if not needed
  ))  

In [ ]:
#delete all the columns that I do not need more
options(repr.matrix.max.cols=50, repr.matrix.max.rows=100)
events_v34 <- events_v33 %>% select(-subject,-`Cell ident`,-context_po,-context_cc,-variation_delete,-variation_delete2,-original_aid_motif1,-original_aid_motif2,-neovariant_aid_motif1,-neovariant_aid_motif2,
                       -neovariant_1_aid_motif,-neovariant_2_aid_motif,
                      -context_po,-context_cc,-vgene_position_aligned)

In [ ]:
events_v34 %>% filter(is.na(codon_cc))#filter(productive == "MIXED", mutation_type == "missense")

In [ ]:
#WriteXLS::WriteXLS(events_v34,
#                    "output/working_table.v3.4.xlsx")

In [ ]:
data_alluvial_v2 <- events_v33 %>% filter(!original_aid_motif %in% c("Not_evaluated", "None")) %>% group_by(original_aid_motif,motif_destroyed) %>% count()
data_alluvial_v2

In [ ]:
AID_colors = c("None" = "#d3d3d3", "RCG" = "#3ac5a4", "WA" = "#3c90be", "WRCY" = "#394abe")

your_data <- data_alluvial_v2

# Define the order you want for original_aid_motif
desired_order <- c("WRCY", "WA", "RCG")

# Set the levels of original_aid_motif according to the desired order
your_data$original_aid_motif <- factor(your_data$original_aid_motif, levels = desired_order)

ao <- ggplot(your_data, 
       aes(axis1 = original_aid_motif, axis2 = motif_destroyed, y = n)) +
  geom_alluvium(aes(fill = color), width = 1/12) +
  geom_stratum(width = 1/12, size = 1, fill = NA,linewidth=0.5) +  # Keep the second stratum transparent
#  scale_fill_brewer(palette = "Set1") +
  theme_minimal() +
  theme(
    legend.position = "none",
    axis.text.y = element_blank(),
    axis.text.x = element_blank(),
    axis.title = element_blank(),
    axis.ticks = element_blank(),
    panel.grid = element_blank()
  )  +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3)

# ggsave("figs_paper/alluvial_original_motif.png", plot = ao, width = 7, height = 7, units = "cm", dpi = 300)
# ggsave("figs_paper/alluvial_original_motif.pdf", plot = ao, width = 7, height = 7, units = "cm")

In [ ]:
data_alluvial_v3 <- events_v33 %>% filter(!neovariant_aid_motif %in% c("Not_evaluated", "None")) %>% group_by(neovariant_aid_motif,motif_destroyed_neovariant) %>% count()
data_alluvial_v3

In [ ]:
# Assign colors directly in the dataframe
data_alluvial_v3$color <- case_when(
  data_alluvial_v3$neovariant_aid_motif == "RCG" ~ "#3ac5a4",
  data_alluvial_v3$neovariant_aid_motif == "WA" ~ "#3c90be",
  data_alluvial_v3$neovariant_aid_motif == "WRCY" ~ "#394abe",
  TRUE ~ NA_character_  # Default, can be transparent (NA) or any color
)

your_data <- data_alluvial_v3

# Define the order you want for original_aid_motif
desired_order <- c("WRCY", "WA", "RCG")

# Set the levels of original_aid_motif according to the desired order
your_data$neovariant_aid_motif <- factor(your_data$neovariant_aid_motif, levels = desired_order)

ao <- ggplot(your_data, 
       aes(axis1 = neovariant_aid_motif, axis2 = motif_destroyed_neovariant, y = n)) +
  geom_alluvium(aes(fill = color), width = 1/12) +
  geom_stratum(width = 1/12, size = 1, fill = NA,linewidth=0.5) +  # Keep the second stratum transparent
#  scale_fill_brewer(palette = "Set1") +
  theme_minimal() +
  theme(
    legend.position = "none",
    axis.text.y = element_blank(),
    axis.text.x = element_blank(),
    axis.title = element_blank(),
    axis.ticks = element_blank(),
    panel.grid = element_blank()
  )  +
  geom_text(stat = "stratum", aes(label = after_stat(stratum)), size = 3)

# ggsave("figs_paper/alluvial_neovariant_motif.png", plot = ao, width = 7, height = 7, units = "cm", dpi = 300)
# ggsave("figs_paper/alluvial_neovariant_motif.pdf", plot = ao, width = 7, height = 7, units = "cm")


### Neovariants data (save some code)

In [ ]:
AID_motif_summ <- events_v33 %>% filter(!is.na(original)) %>% group_by(original_aid_motif) %>% summarise(n= n()) %>% mutate(Percentage= n * 100 / sum(n)) %>% 
mutate(AID=case_when(original_aid_motif == "None" ~ "No",
                          TRUE ~ "Yes"))  %>%
                 dplyr::rename("Motifs"="original_aid_motif") %>% mutate(group = "original")
AID_motif_summ

In [ ]:
print('% AID-related motifs in FL neovariants')
AID_motif_summ %>% 
  filter(Motifs != "None") %>%
  summarise(total_percentage = sum(Percentage)) %>% pull(total_percentage)

In [ ]:
AID_motif_summ_n <- events_v33 %>% filter(!is.na(original)) %>% group_by(neovariant_aid_motif) %>% summarise(n= n()) %>% mutate(Percentage= n * 100 / sum(n)) %>% 
mutate(AID=case_when(neovariant_aid_motif == "None" ~ "No",
                          TRUE ~ "Yes"))  %>%
                 dplyr::rename("Motifs"="neovariant_aid_motif")  %>% mutate(group = "neovariant")
AID_motif_summ_n

In [ ]:
print('% AID-related motifs in FL neovariants')
AID_motif_summ_n %>% 
  filter(Motifs != "None") %>%
  summarise(total_percentage = sum(Percentage)) %>% pull(total_percentage)

In [ ]:
AID_motif_all <- AID_motif_summ %>% bind_rows(AID_motif_summ_n)

In [ ]:
AID_motif_all

In [ ]:
purpleish = c("#B4617A", "#7266ae","#a09ece","#d9dbd5")

In [ ]:

 df <- AID_motif_all
# Convert to factor to ensure the order of the bars
df$group <- factor(df$group, levels = c("original", "neovariant"))

# Create the stacked bar plot
p <- ggplot(df, aes(x = group, y = Percentage, fill = Motifs)) +
  geom_bar(stat = "identity",color="black",linewidth=0.1) +
  scale_fill_manual(values = c("None" = "#d3d3d3", "RCG" = "#3ac5a4", "WA" = "#3c90be", "WRCY" = "#394abe")) +
  theme_minimal() +
  theme(text = element_text(color = "black"),
    axis.title.x = element_text(size = 7, face = "bold",color = "black"),
    axis.title.y = element_text(size = 5, face = "bold",color = "black"),
    axis.text.x = element_text(angle = 45, hjust = 1, size = 7,color = "black"),
    axis.text.y = element_text(size = 5),
    legend.title = element_text(size = 6),
    legend.text = element_text(size = 5),
    legend.box.margin = margin(0, 0, 0, 0),  # Reduce margin around legend
    axis.line = element_line(linewidt = 0.3, colour = "black", linetype=1),
    axis.ticks = element_line(size = 0.3, color="black"),
  ) +
  labs(fill = "Motifs", x = "", y = "Percentage") +
  guides(fill = guide_legend(title.position = "top"))

# Remove the plot title
p <- p + theme(plot.title = element_blank())

# Increase the legend size (this is a bit tricky in ggplot2 but can be done through theme updates)
p <- p + theme(legend.key.size = unit(0.4, 'lines'))

# Adjust the size of the saved plot
ggsave("figs_paper/AID_bar_plot.png", plot = p, width = 7, height = 7, units = "cm", dpi = 300)
ggsave("figs_paper/AID_bar_plot.pdf", plot = p, width = 7, height = 7, units = "cm")

# Print the plot to the R console
print(p)


### IG from bulk data (mutational signature paper) 26 FL samples (-7 LUMC FL)

In [ ]:
mutations_ig <- read.csv("../FL-CLL-MBL_filter/new_analysis_2020/ig_subset/data/df_ct_context.csv") %>% filter(lymph == "FL") %>% mutate(variation=paste0(ref,"-",alt))

In [ ]:
head(mutations_ig)

#### Include AID motifs (check from here, which code to identify motfs I should use?)

In [ ]:
mutations_ig$aid_motif <- mapply(identify_aid_patterns, mutations_ig$variation, mutations_ig$context2)

#### Filter out 7 samples from FL (present in single cell data) and rank motifs

In [ ]:
# filter out 7 samples from FL: 'FL_10000_LN','FL_10971_LN','FL_11770_LN','FL_12118_LN','FL_12282_LN','FL_8934_LN','FL_8934_PBL'
mutations_ig <- mutations_ig %>% filter(!sample %in% c('FL_10000_LN','FL_10971_LN','FL_11770_LN','FL_12118_LN','FL_12282_LN','FL_8934_LN','FL_8934_PBL'))

In [ ]:
count_and_get_perc <- function(data, count_vars, group_vars) {
    counts <- data %>%
      dplyr::count(!!!count_vars) %>%
      dplyr::group_by(!!!group_vars) %>%
        dplyr::mutate(perc = n * 100 / sum(n)) %>%
      as.data.frame()
  
    return(counts)
}

count_aid_motifs <- function(data) {
  motif_counts <- count_and_get_perc(data,
                                     quos(subject, aid_motif1),
                                     quos(subject)) %>%
    dplyr::mutate(aid_motif = factor(aid_motif1,
                                     levels = c("WRCY", "WA", "RCG", "None")))

  return(motif_counts)
}

In [ ]:
AID_motif_wes <-mutations_ig %>%  group_by(aid_motif) %>% summarise(n= n()) %>% mutate(Percentage= n * 100 / sum(n)) %>% 
                mutate(AID=case_when(is.na(aid_motif) ~ "No",
                          TRUE ~ "Yes"))  %>%
                 rename("aid_motif"="Motifs")
AID_motif_wes

In [ ]:
print('% AID-related motifs in FL bulk')
(AID_motif_wes$Percentage[1] + AID_motif_wes$Percentage[2] + AID_motif_wes$Percentage[3])

### IG from bulk data from Marcelo data

In [ ]:
mutations_ig_m <- read.csv("input/igh_sanger_variant_analysis.csv")

In [ ]:
mutations_ig_m <- mutations_ig_m %>% mutate(variation = paste0(ref,"-",alt)) 

In [ ]:
head(mutations_ig_m)

In [ ]:
mutations_ig_m$aid_motif1 <- mapply(identify_aid_patterns, mutations_ig_m$variation, mutations_ig_m$context)
mutations_ig_m$aid_motif2 <- mapply(identify_aid_patterns, reverse(mutations_ig_m$variation), mutations_ig_m$context)

#### Rank motifs

In [ ]:
# filter out 7 samples from FL: 'FL_10000_LN','FL_10971_LN','FL_11770_LN','FL_12118_LN','FL_12282_LN','FL_8934_LN','FL_8934_PBL'
mutations_ig_m <- mutations_ig_m %>% mutate(AIDmotif = paste0(aid_motif1,",",aid_motif2)) %>%
  mutate(aid_motif = case_when(
    aid_motif1 == "WRCY" | aid_motif2 == "WRCY" ~ "WRCY",
    aid_motif1 == "WA" | aid_motif2 == "WA" ~ "WA",
    aid_motif1 == "RCG" | aid_motif2 == "RCG" ~ "RCG",
    TRUE ~ NA
  )) 

In [ ]:
AID_motif_m <-mutations_ig_m %>%  group_by(aid_motif) %>% summarise(n= n()) %>% mutate(Percentage= n * 100 / sum(n)) %>% 
                mutate(AID=case_when(is.na(aid_motif) ~ "No",
                          TRUE ~ "Yes"))  %>%
                 rename("aid_motif"="Motifs")
AID_motif_m

In [ ]:
AID_motif_m <-mutations_ig_m %>%  group_by(sig) %>% summarise(n= n()) %>% mutate(Percentage= n * 100 / sum(n)) %>% 
                mutate(AID=case_when(sig == "None" ~ "No",
                          TRUE ~ "Yes"))  %>%
                 rename("sig"="Motifs")
AID_motif_m

In [ ]:
print('% AID-related motifs in FL bulk')
(AID_motif_m$Percentage[1] + AID_motif_m$Percentage[2] + AID_motif_m$Percentage[3])

In [ ]:
print('% AID-related motifs in FL bulk')
(AID_motif_m$Percentage[1] + AID_motif_m$Percentage[2] + AID_motif_m$Percentage[3])

#### AID mutations between FL neovariant and FL (bulk)

#### Calculate p-value by motif

In [ ]:
# Calculate total counts of 'n' for each source
total_counts <- AID_motifs %>% 
  group_by(Source) %>% 
  summarise(total_n = sum(n), .groups = "keep")

# Join total counts with the original data
AID_motifs <- AID_motifs %>%
  left_join(total_counts, by = "Source")

# Calculate counts for 'other' motifs
AID_motifs$other_n <- AID_motifs$total_n - AID_motifs$n

# Perform chi-squared test for each motif separately
results <- AID_motifs %>% 
  split(.$Motifs) %>% 
  map_df(~{
    test_result <- chisq.test(matrix(c(.$n, .$other_n), nrow = 2))
    tibble(Motifs = unique(.$Motifs), p_value = test_result$p.value)
  })

# Join p-values with the original data
AID_motifs <- AID_motifs %>%
  left_join(results, by = "Motifs")


In [ ]:
AID_motifs

## 7. Substitution type (spectrum)

### Six Substituion pattern of all variant in 10x data

In [ ]:
#clonotype information for K45678B
clonotype <- read.csv("~/repositories/FL_10X_2/250_Clonotypes/outs/clones_2022_01_18.csv") %>% mutate(cell = gsub(".{2}$", "", barcode)) %>% 
 #            select(source, cell,chain,is_cell,productive,high_confidence,umis,reads,seqConcClone, ,cluster,seqConcCount)  %>% 
             mutate(gene=case_when(chain == "IGH" ~ "HC",
                                     chain == "IGK" | chain == "IGL" ~ "LC",
                                    TRUE ~ "NA"))

In [ ]:
#clonotype information for K123B
clonotype_K13B <- read.csv("~/repositories/FL_10X_2/250_Clonotypes/outs/clonesMulti_2022_06_09.csv") %>% mutate(source=paste0(source,"_",subject))

In [ ]:
# default pairwise alignment
# to use for variable tasks
pairwiseAlignmentPreset <- function( pattern, subject, score ) {
  alignment <- pairwiseAlignment( pattern = pattern,
                                  subject = subject,
                                  type = "local",
                                  substitutionMatrix = nucleotideSubstitutionMatrix( match = 1,
                                                                                     mismatch = 0,
                                                                                     baseOnly = FALSE ),
                                  gapOpening = 5,
                                  gapExtension = 2,
                                  scoreOnly = score )
  return( alignment ) }


# # for K45678B 

# d<- read_csv( "~/repositories/FL_10X_2/250_Clonotypes/outs/clones_2022_01_18.csv" ) %>%
#      filter( source == "K4B" )

# po <- d %>% filter( contigId %like% "PO",chain == "IGH") %>% pull( seq )


# test <- d %>%
#     filter( seqClone == 1,
#     chain == "IGH" ) %>%
#     pull( seq ) %>%
#     unique()

# al<- pairwiseAlignmentPreset( toupper( po ),
#     toupper( test ),
#     score = FALSE )

# mm <- al@pattern@mismatch@unlistData

# subst <- sapply( 1: length( mm ),
#         function( i ){
#         list( substr( po, mm[ i ], mm[ i ]),
#         substr( test, mm[ i ], mm[ i ]) )
#         }) #%>% as.data.frame()

# colnames( subst ) <- mm

# subst <- subst %>% t() %>%
#          as.data.frame() %>%
#         rownames_to_column( var = "pos" ) %>%
#         mutate( subst = paste0( V1,"-", V2 ) )

In [ ]:
perform_alignment_analysis <- function(input_df, output_file = "final_output_distinct.csv",
                                       source_col = 'source', chain_col = 'chain', seqClone_col = 'seqClone',
                                       contigId_col = 'contigId', seq_col = 'seq', score = FALSE) {
  # Initialize an empty data frame to store results
  final_output <- data.frame()
  
  # Fetch unique sources and chains
  unique_sources <- unique(input_df[[source_col]])
  unique_chains <- unique(input_df[[chain_col]])
  
  # Iterate over each unique source, chain, and seqClone
  for (src in unique_sources) {
    for (chn in unique_chains) {
      
      # Fetch the reference sequence (seqClone == 0 and contigId == "PO")
      ref_seq <- subset(input_df, input_df[[source_col]] == src & 
                                  input_df[[chain_col]] == chn & 
                                  input_df[[seqClone_col]] == 0 & 
                                  input_df[[contigId_col]] %like% "PO")[[seq_col]]
      
      if (length(ref_seq) == 0) next # Skip if no reference sequence
      
      # Fetch unique seqClone values, ignoring NA
      seqClones <- na.omit(unique(subset(input_df, input_df[[source_col]] == src & 
                                                  input_df[[chain_col]] == chn)[[seqClone_col]]))
      
      for (scl in seqClones) {
        if (scl == 0) next # Skip reference seqClone
        
        # Fetch the test sequence (ignoring NA and taking only the first sequence for each seqClone)
        test_seq <- na.omit(subset(input_df, input_df[[source_col]] == src & 
                                             input_df[[chain_col]] == chn & 
                                             input_df[[seqClone_col]] == scl)[[seq_col]])[1]
        
        # Perform the alignment
        alignment <- pairwiseAlignmentPreset(toupper(ref_seq), toupper(test_seq), score = score)
        
        # Extract the mismatch positions and bases
        mm <- alignment@pattern@mismatch@unlistData
        subst <- sapply(1:length(mm), function(i) {
          list(substr(ref_seq, mm[i], mm[i]), substr(test_seq, mm[i], mm[i]))
        })
        
        # Format and store the results
        if (length(mm) == 0) next # Skip if no mismatches
        colnames(subst) <- mm
        subst <- as.data.frame(t(subst))
        rownames(subst) <- mm
        subst$pos <- rownames(subst)
        subst$subst <- paste0(subst$V1, "-", subst$V2)
        subst[[source_col]] <- src
        subst[[seqClone_col]] <- scl
        
        # Append to final output
        final_output <- rbind(final_output, subst)
      }
    }
  }
  
  final_output_distinct <- final_output %>%
    distinct(pos, source, subst, .keep_all = TRUE)
  
  # Export the final output
  if (nrow(final_output_distinct) > 0) {
    write.csv(final_output_distinct[, c("pos", "subst", source_col, seqClone_col)], output_file, row.names=FALSE)
  } else {
    print("Final output is empty.")
  }
}


In [ ]:
# Example usage
perform_alignment_analysis(clonotype, output_file = "output/final_output_distinct_K45678B.csv", source_col = "source")

In [ ]:
# Example usage
perform_alignment_analysis(clonotype_K13B, output_file = "output/final_output_distinct_K123B.csv", source_col = "source")

In [ ]:
K123B_subs <- read_csv("output/final_output_distinct_K123B.csv") %>% mutate(subst = toupper(subst))

In [ ]:
K45678B_subs <- read_csv("output/final_output_distinct_K45678B.csv") %>% mutate(subst = toupper(subst))

In [ ]:
six_pattern <- function(df, case_col, output_df_name) {
  case_col_sym <- sym(case_col)

  
  df_transformed <- df %>%
    mutate(
      substitution = case_when(
        !!case_col_sym == "G-T" ~ "C-A",
        !!case_col_sym == "G-C" ~ "C-G",
        !!case_col_sym == "G-A" ~ "C-T",
        !!case_col_sym == "A-T" ~ "T-A",
        !!case_col_sym == "A-G" ~ "T-C",
        !!case_col_sym == "A-C" ~ "T-G",
        TRUE ~ as.character(!!case_col_sym)
      )
    ) %>% filter(!substitution %in% c("A-A","C-C","G-G","T-T"))
  
  assign(output_df_name, df_transformed, envir = .GlobalEnv)
}

In [ ]:
KBall_subs <- K123B_subs %>% bind_rows(K45678B_subs) %>% mutate(
    source = case_when(
      source == "K4B" ~ "K4B_S8934",
      source == "K5B" ~ "K5B_S8934",
      source == "K6B" ~ "K6B_S13530",
      source == "K7B" ~ "K7B_S10000",
      source == "K8B" ~ "K8B_S13553",
      TRUE ~ source
    )) 

six_pattern(
  KBall_subs, 
  case_col = "subst", 
  output_df_name = "KBall_subs_six_pattens"
)

### Group substitution by subject

In [ ]:
KBall_subs_subject <- KBall_subs_six_pattens %>% group_by(source,substitution) %>% count() %>% group_by(source) %>% mutate(Percentage =n * 100 / sum(n)) %>%
                      dplyr::rename(subject=source) %>% mutate(source = "10x_bulk") 
head(KBall_subs_subject,12)

In [ ]:
# Set plot dimensions
options(repr.plot.width=20, repr.plot.height=8)

# Generate a color palette with as many colors as there are variation types
my_palette <- colorRampPalette(brewer.pal(9, "Set1"))(length(unique(KBall_subs_subject$substitution)))

# Create bar plot
ggplot(KBall_subs_subject, aes(x = subject, y = Percentage, fill = substitution)) +
  geom_bar(stat = "identity", position = position_dodge()) +
  theme_minimal() +
  labs(title = "Variation Percentages by Source", x = "Variation", y = "Percentage (%)") +
  scale_fill_manual(values = my_palette) +
  theme(legend.position = "bottom")


### Overall substituion

In [ ]:
#data neovariant
# remove variation where we can not define hierarchy
spectro <- events_v33 %>% filter(!is.na(variation_delete)) %>% mutate(subject=str_replace(subject, "-[^-]+$", "")) 
                    
# transform substituion to six pattern
six_pattern(
  spectro, 
  case_col = "variation_delete", 
  output_df_name = "spectro"
)

spectro_subject <- spectro %>% group_by(subject,substitution) %>% count() %>% group_by(subject) %>% mutate(Percentage =n * 100 / sum(n)) %>% mutate(source = "neoavariants")
spectro_summary <- spectro %>% group_by(substitution) %>% count() %>% ungroup() %>% mutate(Percentage =n * 100 / sum(n)) %>% mutate(source = "neoavariants") %>% dplyr::rename(total_n=n) %>% select(source,everything()) 


In [ ]:
# data from muational signature filter by the subject used in 10x
mutations_ig <- mutations_ig %>%filter(!sample %in% c('FL_10000_LN','FL_10971_LN','FL_11770_LN','FL_12118_LN','FL_12282_LN','FL_8934_LN','FL_8934_PBL')) 

# transform substituion to six pattern
six_pattern(
  mutations_ig, 
  case_col = "variation", 
  output_df_name = "mutations_ig_spectro"
)

#Summary data by substitution
mutations_ig_spectro <- mutations_ig_spectro %>% group_by(substitution) %>% count() %>% ungroup() %>% mutate(Percentage =n * 100 / sum(n)) %>% 
mutate(source = "Ig_WES") %>% dplyr::rename(total_n=n) %>% select(source,everything()) 

In [ ]:
# data sanger Marcelo
# transform substituion to six pattern
six_pattern(
  mutations_ig_m, 
  case_col = "variation", 
  output_df_name = "mutations_ig_m_spectro"
)

# #Summary data by substitution
mutations_ig_m_spectro <- mutations_ig_m_spectro %>% group_by(substitution) %>% filter(substitution %in% mutations_ig_spectro$substitution) %>% 
                          count() %>% ungroup() %>% mutate(Percentage =n * 100 / sum(n)) %>% 
                        mutate(source = "Ig_sanger") %>% dplyr::rename(total_n=n) %>% select(source,everything()) 

In [ ]:
#combine 4 data source 10x_bulk,spectro_summary,mutations_ig_spectro,mutations_ig_m_spectro

spectro_combine <- KBall_subs_six_pattens %>% group_by(substitution) %>% count() %>% ungroup() %>% mutate(Percentage =n * 100 / sum(n)) %>% 
               mutate(source="10x_bulk") %>% dplyr::rename(total_n=n) %>% select(source,everything()) %>%
               bind_rows(spectro_summary,mutations_ig_spectro,mutations_ig_m_spectro )

In [ ]:
head(spectro_combine)

In [ ]:
# Using a cool color palette
my_custom_palette <- c("#5E81AC", "#88C0D0", "#81A1C1", "#8FBCBB", "#B48EAD")

# Create bar plot
ggplot(spectro_combine, aes(x = substitution, y = Percentage, fill = source)) +
  geom_bar(stat = "identity", position = position_dodge()) +
  theme_minimal() +
  labs( x = "Substitution spectrum", y = "Percentage (%)") +
  scale_fill_manual(values = my_custom_palette) +  # Use custom color palette
  theme(legend.position = "bottom",
        text = element_text(size = 16),  # Increase general text size
        axis.text.x = element_text(size = 16),  # Increase x-axis label size
        legend.text = element_text(size = 18),  # Increase legend text size
        plot.title = element_text(size = 16, hjust = 0.5))  # Increase title size and center it



In [ ]:
# WriteXLS::WriteXLS(spectro_combine,
#                     "output/substitution_spectrum_aggregated.xlsx" )

In [ ]:
spectro_combine

In [ ]:
data <- spectro_combine

# Replace underscores in the 'source' labels
data$source <- gsub("_", " ", data$source)

# Reorder the 'substitution' factor according to your specified order for the plot
data$substitution <- factor(data$substitution, levels = c('T-A','C-A','T-G', 'T-C', 'C-G', 'C-T'))
data$source <- factor(data$source, levels = c('Ig sanger','Ig WES','10x bulk', 'neoavariants'))

# Calculate the total counts for each source
totals <- data %>%
  group_by(source) %>%
  summarize(Total_n = sum(total_n), .groups = 'drop')

# Create the stacked bar plot
p <- ggplot(data, aes(x = source, y = Percentage, fill = substitution)) +
  geom_bar(stat = "identity", position = "fill",color="black",linewidth=0.1) +
  #geom_text(data = totals, aes(x = source, label = Total_n), vjust = 0, nudge_y = max(data$Percentage) * 0.05) +
  scale_fill_manual(values = c("#f69b70", "#f9c2a7", "#fde8df", "#b9e8ff", "#1cb6ff", "#0098e0")) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 65, vjust = 0.6, size = 8), 
        axis.title = element_text(size = 10),
        plot.title = element_text(size = 12),
       axis.line = element_line(linewidt = 0.3, colour = "black", linetype=1),
       axis.ticks = element_line(size = 0.3, color="black"),
       legend.key.size = unit(0.3, 'cm')) +
  labs(x = "Data source", y = "Fraction of mutations", fill = "Substitution")



# Save the plot
#ggsave("figs_paper/substitution_spectrum_stacked_barplot.png", plot = p, width = 7, height = 7, units = "cm", dpi = 300)
#ggsave("figs_paper/substitution_spectrum_stacked_barplot.pdf", plot = p, width = 7, height = 7, units = "cm")

# Print the plot
print(p)


In [ ]:
# function to perform chi-square test on each group
perform_chisq_test <- function(df) {
  chisq_result <- chisq.test(df$Percentage)$p.value
  return(data.frame(chisq_test_p_value = chisq_result))
}

# Perform the chi-square tests
chi_square_results <- spectro_combine %>% 
  filter(source %in% c("10x_bulk", "neoavariants")) %>% 
  group_by(substitution, source) %>% 
  group_by(substitution) %>% 
  nest() %>% 
  rowwise() %>% 
  summarise(perform_chisq_test(data))

# Print chi-square test results
print(chi_square_results)


#### Data for Hendrik (substitution by subject)

In [ ]:
spectro_10x <- KBall_subs_subject %>% bind_rows(spectro_subject)

In [ ]:
spectro_10x <- spectro_10x %>% mutate(Patient_id = str_extract(subject, "S\\d+")) %>% group_by(Patient_id,source,substitution) %>%
  summarise(total_n=sum(n)) %>%  mutate(Percentage=total_n * 100 / sum(total_n))

In [ ]:
head(spectro_10x)

In [ ]:
# WriteXLS::WriteXLS(spectro_10x,
#                     "output/substitution_spectrum_10x.xlsx" )

In [ ]:
data <- spectro_10x

# Replace underscores in the 'source' labels
data$source <- gsub("_", " ", data$source)

# Reorder the 'substitution' factor according to your specified order for the plot
data$substitution <- factor(data$substitution, levels = c('T-A','C-A','T-G', 'T-C', 'C-G', 'C-T'))

# Calculate the total counts for each source
totals <- data %>%
  group_by(Patient_id) %>%
  summarize(Total_n = sum(total_n), .groups = 'drop')

# Create the stacked bar plot
p <- ggplot(data %>% filter(Patient_id %in% c("S10000","S13530","S13553", "S144")), aes(x = source, y = Percentage, fill = substitution)) +
  geom_bar(stat = "identity", position = "fill",color="black",linewidth=0.1) +
  #geom_text(data = totals, aes(x = source, label = Total_n), vjust = 0, nudge_y = max(data$Percentage) * 0.05) +
  scale_fill_manual(values = c("#f69b70", "#f9c2a7", "#fde8df", "#b9e8ff", "#1cb6ff", "#0098e0")) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 65, vjust = 0.6, size = 8), 
        axis.title = element_text(size = 10),
        plot.title = element_text(size = 12),
       axis.line = element_line(linewidt = 0.3, colour = "black", linetype=1),
       axis.ticks = element_line(size = 0.3, color="black"),
       legend.key.size = unit(0.3, 'cm'),
       panel.spacing = unit(1, "lines")) +
  labs(x = "Data source", y = "Fraction of mutations", fill = "Substitution") +
  facet_grid(. ~ Patient_id)



# Save the plot
#ggsave("figs_paper/substitution_spectrum_stacked_barplot.png", plot = p, width = 7, height = 7, units = "cm", dpi = 300)
ggsave("figs_paper/substitution_spectrum_stacked_barplot_subject.pdf", plot = p, width = 10.5, height = 7, units = "cm")

# Print the plot
print(p)


## 8. Calculate combinations of variants

In [ ]:
events_v31 <- readxl::read_xlsx("output/events.v3.1.xlsx")

In [ ]:
#Filter events in CD3 and FR4
df_umis <- read_csv("output/df_umis.csv") 

In [ ]:
real_pos <- df_summary %>% filter(!subregion %in% c("CDR3", "FR4")) %>% select(subject, cell,position)

In [ ]:
df_umis_filter <- df_umis %>% inner_join(real_pos, by=c("subject", "cell", "position"))

In [ ]:
# Your existing script
df_combinations <- df_umis_filter %>%
  unite("position_nucl", c("position", "nucl"), sep = ":", remove = FALSE) %>%
  unite("position_nucl_po", c("position", "nucl_po"), sep = ":", remove = FALSE) %>%
  group_by(subject, cell, umi) %>%
 summarise(nucl_combination = toString(position_nucl),nucl_combination_po = toString(position_nucl_po), .groups = 'drop') %>%
  ungroup()

# Calculate the number of positions in each nucl_combination
df_combinations <- df_combinations %>%
  mutate(position_count = str_count(nucl_combination, pattern = ",") + 1)

# Calculate the count of distinct UMIs for each nucl_combination in each cell
df_combinations <- df_combinations %>%
  group_by(subject, cell, nucl_combination) %>%
  mutate(umi_count = n_distinct(umi)) %>%
  ungroup()

# Determine the maximum number of positions for each cell
max_positions_per_cell <- df_combinations %>%
  group_by(subject, cell) %>%
  summarise(max_positions = max(position_count)) %>%
  ungroup()

# Filter to only include nucl_combinations with the max amount of positions and more than one position
df_filtered <- df_combinations %>%
  inner_join(max_positions_per_cell, by = c("subject", "cell")) %>%
  filter(position_count == max_positions, position_count > 1) %>% select(-umi,-max_positions) %>% unique()


In [ ]:
df_filtered %>% filter(cell == "AAAGCAAGTTCAGACT")

In [ ]:
# Count the number of unique nucl_combination for each cell
df_cell_combination_counts <- df_filtered %>%
  group_by(subject, cell) %>%
  summarise(unique_combinations = n_distinct(nucl_combination)) %>%
  ungroup()

# Filter for cells with more than two unique nucl_combination
df_cells_with_multiple_combinations <- df_filtered %>%
  inner_join(df_cell_combination_counts, by = c("subject", "cell")) %>%
  filter(unique_combinations > 1)


In [ ]:
# Identify the groups "Neo", "PO" and "PO-Neo"
df_umi_combination <- df_cells_with_multiple_combinations %>% 
  mutate(
   group = mapply(function(nucl, nucl_po) {
      # Split the strings into individual components
      nucl_split <- strsplit(nucl, ", ")[[1]]
      nucl_po_split <- strsplit(nucl_po, ", ")[[1]]

      # Ensure that both vectors have the same length
      if (length(nucl_split) != length(nucl_po_split)) {
        return(NA)
      }

      # Count matches and mismatches
      matches <- sum(nucl_split == nucl_po_split)
      mismatches <- length(nucl_split) - matches

      # Determine the category based on counts
      if (matches == length(nucl_split)) {
        return("PO")
      } else if (mismatches == length(nucl_split)) {
        return("Neo")
      } else {
        return("PO-Neo")
      }
    }, nucl_combination, nucl_combination_po)
  )

In [ ]:
df_umi_combination <- df_umi_combination %>%
  group_by(subject, cell) %>%
  # Check if both PO and Neo are present
  mutate(both_po_neo_present = all(c("PO", "Neo") %in% unique(group))) %>%
  # Order by umi_count within each group and get the top two group names
  arrange(desc(umi_count)) %>%
  mutate(top_two_groups = list(head(unique(group), 2))) %>%
  # Check if the top two groups are PO and Neo
  mutate(is_expected = both_po_neo_present & all(c("PO", "Neo") %in% top_two_groups[[1]])) %>%
  # Assign type based on the condition
  mutate(type = if_else(is_expected, "expected", "unexpected")) %>%
  # Optionally, remove the helper columns
  select(-both_po_neo_present, -top_two_groups, -is_expected)

# View the result
head(df_umi_combination)

In [ ]:

# Assuming df is your data frame
df_umi_combination_w <- df_umi_combination %>%
  group_by(subject, cell) %>%
  mutate(
    highest_umi_count = umi_count == max(umi_count)
  ) %>%
  ungroup()

# Function to check if two combinations are completely different
all_positions_different <- function(comb1, comb2) {
  positions1 <- unlist(strsplit(comb1, ", "))
  positions2 <- unlist(strsplit(comb2, ", "))
  all(mapply(function(p1, p2) substr(p1, nchar(p1), nchar(p1)) != substr(p2, nchar(p2), nchar(p2)), positions1, positions2))
}

# Apply the function to compare nucl_combinations
df_umi_combination_w <- df_umi_combination_w %>%
  group_by(subject, cell) %>%
  mutate(
    alternative_type = case_when(
      highest_umi_count ~ "highest_umi_count",
      sapply(nucl_combination, function(nc) all_positions_different(nc, nucl_combination[highest_umi_count])) ~ "all positions alternative",
      TRUE ~ "partial alternative"
    )
  ) %>%
  ungroup()

# Separate partial alternatives and rank them
partial_alternatives <- df_umi_combination_w %>%
  filter(alternative_type == "partial alternative") %>%
  group_by(subject, cell) %>%
  arrange(desc(umi_count)) %>%
  mutate(alternative_rank = row_number()) %>%
  ungroup()

# Merge the ranked partial alternatives back into the main dataframe
df_umi_combination_wa <- df_umi_combination_w %>%
  left_join(partial_alternatives %>% select(subject, cell, nucl_combination, alternative_rank), by = c("subject", "cell", "nucl_combination")) %>%
  mutate(
    alternative_type = ifelse(alternative_type == "partial alternative" & !is.na(alternative_rank),
                              paste("partial alternative", alternative_rank),
                              alternative_type)
  ) %>%
  select(-alternative_rank,-highest_umi_count)  # Optional: remove the helper column


In [ ]:
# WriteXLS::WriteXLS(df_umi_combination_wa,
#                     "output/umi_combinations_updated.xlsx" )


In [ ]:
# Function to compare two nucleotide combinations
compare_combinations <- function(comb1, comb2) {
  # Split the combinations into individual positions
  positions1 <- strsplit(comb1, ", ")[[1]]
  positions2 <- strsplit(comb2, ", ")[[1]]
  
  # Compare the positions
  matches <- mapply(function(p1, p2) p1 == p2, positions1, positions2)
  
  # Calculate percentage of match
  percent_match <- sum(matches) / length(matches) * 100
  return(percent_match)
}

# Applying the function to the dataframe
df_umi_combination_wat <- df_umi_combination_wa %>%
  rowwise() %>%
  mutate(
    match_percentage_po = compare_combinations(nucl_combination, nucl_combination_po)
  ) %>%
  ungroup()


In [ ]:
#Updating labeling for plotting
df_umi_combination_wate <- df_umi_combination_wat %>%
  group_by(subject, cell) %>%
  mutate(
    highest_count = sum(alternative_type == "highest_umi_count"),
    label = case_when(
      highest_count == 2 & alternative_type == "highest_umi_count" & match_percentage_po > 50 ~ "Highest",
      highest_count == 2 & alternative_type == "highest_umi_count" & group != "PO" ~ "Opposite",
      highest_count != 2 & alternative_type == "highest_umi_count" ~ "Highest",
      alternative_type == "all positions alternative" ~ "Opposite",
      TRUE ~ as.character(alternative_type)  # Keep current label or assign a default
    )
  )   %>%
  group_by(subject,cell) %>%
  mutate(total_umi = sum(umi_count),
         umi_percentage = (umi_count / total_umi) * 100)
# Create a column for subject type identifier (HC or LC)
df_umi_combination_wate$subject_type <- ifelse(grepl("HC$", df_umi_combination_wate$subject), "HC", "LC")

In [ ]:
#Which cells do not have Hightest and opposite

# Group the data by 'subject' and 'cell'
groups <- df_umi_combination_wate %>%
  group_by(subject, cell) %>%
  summarise(alternative_types = list(unique(label)))

# Check that each group has both 'Highest' and 'Opposite'
verification <- groups %>%
  rowwise() %>%
  mutate(contains_both = all(c("Highest", "Opposite") %in% alternative_types)) %>%
  ungroup()

# Verify if all groups have both 'Highest' and 'Opposite'
all_groups_have_both <- all(verification$contains_both)

# If not all groups have both, show which do not
groups_without_both <- verification %>%
  filter(!contains_both)

# You can view the results with
print(all_groups_have_both)
print(groups_without_both)

#Conclusion: 4 of them have only highest and not opposite, which is something that can happens, 
#Cells where is not possible to define with one is highest (2 opposite), it will remove for the umi_combination analysis 
# 1. TGACGGCGTGTGAATA
# 3. "GGGAGATAGGACTGGT"
# 5. TGATTTCTCACAAACC
# 6. CCTTCGATCCTGTAGA
# 10. GAAATGAAGGGATCTG
# 11. GGACAGAAGAAGCCCA
# 12 TGGGAAGAGATCCGAG

In [ ]:
#Case type 1: one highest and none opposite: this cell has a odd pattern when I check the image, I think should be discarted by statidtics but I am not filter out
df %>% arrange(subject,cell) %>% filter(cell == "TTGGCAAGTTTGGCGC")

In [ ]:
#Case type 2: only opposite ( 7 cells/chain)
df %>% arrange(subject,cell) %>% filter(cell == "TGGGAAGAGATCCGAG")

In [ ]:
# filter out cells with only opposite (because, it is not possible to identified the highest)

df_umi_combination_wate <- df_umi_combination_wate %>% 
                            filter(!cell %in% c("GGGAGATAGGACTGGT", "TGACGGCGTGTGAATA","GGGAGATAGGACTGGT","TGATTTCTCACAAACC","CCTTCGATCCTGTAGA","GAAATGAAGGGATCTG","GGACAGAAGAAGCCCA","TGGGAAGAGATCCGAG")) 

df <- df_umi_combination_wate 

In [ ]:
# WriteXLS::WriteXLS(df_umi_combination_wate,
#                     "output/umi_combinations_v2.xlsx" )

In [ ]:
# Provided color codes
panel_A_colors <- c("#f69b70", "#f9c2a7", "#fde8df", "#b9e8ff", "#1cb6ff", "#0098e0")
panel_B_colors <- c( "#d3d3d3", "#3ac5a4", "#3c90be", "#394abe")

# Select distinct colors for "Highest" and "Opposite"
highest_color <- "#46a4d8" # A strong blue
opposite_color <- "#c2d6ed"   

# Select a base color for the "partialalt" labels and generate shades

partialalt_shades <- colorRampPalette(c("#adadad", "#f9f9f9"))(5)  # Assuming 4 partialalt labels

# Define the custom color palette
custom_colors <- c("Highest" = highest_color, "Opposite" = opposite_color, 
                   setNames(partialalt_shades, paste("partial alternative", 1:5)))

# Define colors for HC and LC
hc_lc_colors <- c("HC" = "#39bc9f", "LC" = "#ace6e3")

# Add subject_type_color column based on the 'subject' ending 
df <- df  %>% 
   mutate(cell_chain=paste0(cell,"_",subject_type))

summarized_data <- df %>% 
  group_by(cell_chain) %>%
  filter(label == "Highest" | label == "Opposite") %>%
  summarise(total_umi = sum(umi_percentage)) %>%
  arrange(desc(total_umi))

# Adjust the levels of the factor for 'cell' based on 'total_umi'
df$cell_chain <- factor(df$cell_chain, levels = summarized_data$cell_chain)

# Create the plot
ggplot_object <- ggplot(df, aes(x = cell_chain, y = umi_percentage, fill = label)) + #reorder(cell, cell_order)
  geom_bar(stat = "identity", position=position_stack(reverse = TRUE)) +
  scale_fill_manual(values = custom_colors) +
  geom_point(data = df, aes(x = cell_chain, y = 102, color = subject_type), 
             size = 2, shape = 15) +  # Using square shape for the points
  scale_color_manual(values = hc_lc_colors) +
  labs(x = "", y = "Percentage of UMI Count") +
  theme_minimal() +
  theme(axis.text.x = element_blank(),
        legend.position = "bottom",
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank()) +
  guides(fill = guide_legend(title = "Label", title.position = "top")) +
  scale_y_continuous(limits = c(0, 140), breaks = seq(0, 124, by = 25)) +
  geom_hline(yintercept = 95, linetype = "33", color = "#595959") 

# Calculate the summary for position_count
summary_position_count <- df %>%
  group_by(cell_chain) %>%
  summarise(avg_position_count = mean(position_count))  # or use sum, max, etc.

# Adjust the levels of the factor for 'cell' based on 'total_umi'
summary_position_count$cell_chain <- factor(summary_position_count$cell_chain, levels = summarized_data$cell_chain)

small_plot <- ggplot(summary_position_count, aes(x = cell_chain, y = avg_position_count)) +
  geom_bar(stat = "identity") +
   scale_y_continuous(limits = c(0, 10), 
                     breaks = seq(0, 10, by = 5)) +  # Adjust y-axis limits and expansion
  theme_minimal() +  # Use a minimal theme as a base
  theme(
    axis.title.x = element_blank(),  # Remove x-axis title
    axis.title.y = element_blank(),
    axis.text.x = element_blank(),  # Remove x-axis text
    axis.ticks.x = element_blank(),  # Remove x-axis ticks
    plot.margin = margin(1, 0, 1, -0.35, "cm"),
            panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank(),
  panel.grid.major.y = element_line(),  # Remove major y-axis grid lines
    panel.grid.minor.y = element_blank()  # Remove minor y-axis grid lines
) +  scale_y_continuous(limits = c(0, max(summary_position_count$avg_position_count)),  # Set y-axis limits dynamically
                     breaks = seq(0, max(summary_position_count$avg_position_count), 2))   # More breaks on y-axis

# Create a grob (graphic object) from the small plot
small_grob <- ggplotGrob(small_plot)

# Embed the small plot on top of the main plot
ggplot_object <- ggplot_object +
  annotation_custom(grob = small_grob, xmin = -Inf, xmax = Inf, ymin = 85, ymax = 140) 

# Print the plot
print(ggplot_object)

# ggsave("figs_paper/umi_combination.pdf", plot = ggplot_object, width = 17.5, height = 12, units = "cm")

In [ ]:

# Function to split the combination into position-letter pairs
split_combination <- function(combination) {
  str_split(combination, ",\\s*")[[1]] %>%
    map(~str_split(., ":")[[1]][2])  # Extract only the nucleotide letters
}

# Function to compare combinations and count matches
compare_and_count_matches <- function(highest_comb, other_comb) {
  # Split into position-letter pairs
  highest_pairs <- split_combination(highest_comb)
  other_pairs <- split_combination(other_comb)

  # Count matches
  sum(mapply(`==`, highest_pairs, other_pairs))
}

# Process the data
comparison_results <- df %>%
  group_by(subject, cell) %>%
  filter(any(label == "Highest")) %>%
  reframe(
    highest_combination = nucl_combination[label == "Highest"][1],
    other_combination = c(nucl_combination[label == "Highest"][1], nucl_combination[label != "Highest"]),
            unique_combinations = unique_combinations[label == "Highest"],
    match_count = c(length(split_combination(nucl_combination[label == "Highest"][1])), 
                    map_int(nucl_combination[label != "Highest"], 
                            ~compare_and_count_matches(nucl_combination[label == "Highest"][1], .))),
    position_count = position_count[label == "Highest"],  # Retain position_count for the 'Highest'
      umi_percentage = umi_percentage,
      cell_chain=cell_chain
  ) %>%
  ungroup()

In [ ]:
# View the results
comparison_results <- comparison_results %>% mutate(ummatch = (position_count-match_count)*1/position_count) 
head(comparison_results)

In [ ]:

# Sample code for creating a dot plot
ggplot_object <- ggplot(comparison_results, aes(x = cell_chain, y = ummatch, size = umi_percentage)) +
  geom_point(alpha = 0.7, shape=1) +  # Adjust alpha for dot transparency if needed
  scale_size_continuous(name = "Size Variable", range = c(0.1, 5)) +  # Adjust the size range as needed
  labs(x = "cell", y = "unmatch") +
    theme_minimal() +
  theme(axis.text.x = element_blank(),
        legend.position = "bottom",
        panel.grid.major.x = element_blank(),
        panel.grid.minor.x = element_blank()) 

# Print the plot
print(ggplot_object)


ggsave("figs_paper/umi_combination_dotplot.pdf", plot = ggplot_object, width = 17.5, height = 7, units = "cm")

In [ ]:
# new visualization

result <- df %>% group_by(cell_chain) %>%   summarize(
    sum_umi_percentage = sum(umi_percentage[(label == "Highest" | label == "Opposite")], na.rm = TRUE)
  )
head(result)

In [ ]:

# Creating the plot
dis <- ggplot(result, aes(x = sum_umi_percentage)) +
  geom_histogram(binwidth = 2, fill = "blue", color = "black", alpha = 0.7) + # or geom_density() for a density plot
  geom_vline(xintercept = 90, color = "red", linetype = "dashed", size = 1) +
#geom_text(data = annotations, aes(x = midpoint, y = count, label = count), vjust = -0.5) +
  theme_minimal() +
  labs(title = "Distribution of Summed UMI Percentage",
       x = "Sum of UMI Percentage",
       y = "Frequency") +
scale_x_continuous(breaks = seq(30,100, by = 5))

dis
ggsave("figs_paper/umi_combination_distribution.pdf", plot = dis, width = 10, height = 7, units = "cm")

#### New PO classification

In [ ]:
# Step 1: Identify groups with at least 'Highest' and 'Opposite' labels
valid_groups <- df %>%
  group_by(subject, cell) %>%
  filter(any(label == "Highest") & any(label == "Opposite")) %>%
  ungroup() %>%
  distinct(subject, cell)

# Step 2, 3, 4, and 5: Categorize each row
df <- df %>%
  rowwise() %>%
  mutate(categorized_label = if_else(
    (subject %in% valid_groups$subject) & (cell %in% valid_groups$cell),
    if_else(
      label %in% c("Highest", "Opposite"),
      {
        # Find the max match_percentage_po for 'Highest' and 'Opposite'
        max_po_highest <- max(df$match_percentage_po[df$subject == subject & df$cell == cell & df$label == "Highest"], na.rm = TRUE)
        max_po_opposite <- max(df$match_percentage_po[df$subject == subject & df$cell == cell & df$label == "Opposite"], na.rm = TRUE)

        if (max_po_highest == max_po_opposite && max_po_highest == 50) {
          "Undetermined"
        } else if (label == "Highest" && max_po_highest >= max_po_opposite) {
          "PO"
        } else if (label == "Opposite" && max_po_opposite > max_po_highest) {
          "PO"
        } else {
          "Neovariant"
        }
      },
      NA_character_
    ),
    "invalid"
  ))

In [ ]:
# Pivoting the data into a wide format

df <- df %>% group_by(subject, cell) %>%
  filter(!is.na(categorized_label), label %in% c("Highest", "Opposite")) %>%
  mutate(index = row_number()) %>%
  ungroup()

df_wide <- df %>% filter(!is.na(categorized_label)) %>% select(subject,cell,nucl_combination_po,nucl_combination,match_percentage_po,categorized_label,index) %>% 
  pivot_wider(
    id_cols = c(subject, cell,nucl_combination_po),
    names_from = index,
    names_sep = "",
    values_from = c(nucl_combination, categorized_label, match_percentage_po)
  )

In [ ]:
# Function to expand the rows based on nucl_combinations to fit to master table format
expand_rows <- function(row) {
  n1 <- strsplit(as.character(row$nucl_combination1), ", ")[[1]]
  n2 <- strsplit(as.character(row$nucl_combination2), ", ")[[1]]
  
  data.frame(subject = row$subject, 
             cell = row$cell, 
             nucl_combination_po = row$nucl_combination_po, 
             nucl_combination_1 = row$nucl_combination1, 
             nucl_combination_2 = row$nucl_combination2, 
             nucl_combination1 = n1, 
             nucl_combination2 = n2, 
             categorized_label1 = row$categorized_label1, 
             categorized_label2 = row$categorized_label2, 
             match_percentage_po1 = row$match_percentage_po1, 
             match_percentage_po2 = row$match_percentage_po2)
}

# Applying the function to each row
df_expanded <- do.call(rbind, lapply(1:nrow(df_wide), function(i) expand_rows(df_wide[i, ])))

# Function to split nucleotide combinations into position and nucleotide
split_nucl_combination <- function(comb) {
  parts <- strsplit(comb, ":")[[1]]
  return(parts)
}

# Applying the function and expanding the DataFrame
df_final <- df_expanded %>%
  rowwise() %>%
  mutate(pos= as.numeric(split_nucl_combination(nucl_combination1)[1]),
         nucl1 = split_nucl_combination(nucl_combination1)[2],
         nucl2 = split_nucl_combination(nucl_combination2)[2]) %>% select(-nucl_combination1,-nucl_combination2) %>%
  ungroup()

In [ ]:
head(df_final)

In [ ]:
#pivot longer df to match event_v3

# Pivoting nucl and categorized_label columns simultaneously
df_longer <- df_final %>%
  pivot_longer(
    cols = starts_with("nucl"),
    names_to = "nucl_type",
    values_to = "nucl",
    names_pattern = "nucl(\\d)"
  ) %>%
  pivot_longer(
    cols = starts_with("categorized_label"),
    names_to = "categorized_label_type",
    values_to = "categorized_label",
    names_pattern = "categorized_label(\\d)"
  ) %>%
  filter(sub(".*_", "", nucl_type) == sub(".*_", "", categorized_label_type)) %>%
  select(-nucl_type, -categorized_label_type,-match_percentage_po1,-match_percentage_po2)

In [ ]:
df_longer %>% pull(categorized_label) %>% table()

In [ ]:
#create the columns original and neovariant1

# Creating new columns 'po' and 'neovariant' based on 'categorized_label'
df_transformed <- df_longer %>%
  group_by(subject, cell, pos) %>%
  mutate(undetermined_count = cumsum(categorized_label == "Undetermined"),
         original = ifelse(categorized_label == "PO", nucl, NA_character_),
         neovariant1 = ifelse(categorized_label == "Neovariant" | categorized_label == "unvalid" |(categorized_label == "Undetermined" & undetermined_count == 1), nucl, NA_character_),
         neovariant2 = ifelse(categorized_label == "Undetermined" & undetermined_count == 2, nucl, NA_character_)) %>%
  ungroup() %>%
  select(subject, cell, pos, original, neovariant1, neovariant2) %>%
  group_by(subject, cell, pos) %>%
  summarize(
    original = max(original, na.rm = TRUE),
    neovariant1 = max(neovariant1, na.rm = TRUE),
    neovariant2 = max(neovariant2, na.rm = TRUE),
    .groups = "drop"
  ) %>% dplyr::rename("position"="pos")

In [ ]:
head(df_transformed) #%>% filter(cell == "CGTTGGGTCCTTTCTC")

In [ ]:
df_transformed %>% group_by(subject) %>% summarize(different_cells = n())# %>% pull(different_cells) %>% sum()

In [ ]:
df_transformed %>% group_by(subject) %>% summarize(different_cells = n()) %>% pull(different_cells) %>% sum()

In [ ]:
options(repr.matrix.max.cols=50, repr.matrix.max.rows=100)
head(events_v31)

In [ ]:
# Do not forget to annotate the invalid cell, the ones that have only one nucl (20 cells/gene)
#subset_n_event %>% filter(rowSums(is.na(select(., original_umis.y, neovariant1_umis.y, neovariant2_umis.y))) > 1)

In [ ]:
#969 events
multiple_events <- events_v31 %>% inner_join(df_transformed %>% 
                          select(subject,cell,position,original,neovariant1,neovariant2), by=c("subject","cell","position")) %>% 
                          mutate(original_umis.y = case_when(original.y == original.x ~ original_umis,
                                                             original.y == neovariant1.x ~ neovariant1_umis,
                                                             original.y == neovariant2.x ~ neovariant2_umis,
                                                             TRUE ~ NA),
                                 neovariant1_umis.y = case_when(neovariant1.y == original.x ~ original_umis,
                                                             neovariant1.y == neovariant1.x ~ neovariant1_umis,
                                                             neovariant1.y == neovariant2.x ~ neovariant2_umis,
                                                             TRUE ~ NA),
                                 neovariant2_umis.y = case_when(neovariant2.y == original.x ~ original_umis,
                                                             neovariant2.y == neovariant1.x ~ neovariant1_umis,
                                                             neovariant2.y == neovariant2.x ~ neovariant2_umis,
                                                             TRUE ~ NA)) %>%
                 select(-original.x,-neovariant1.x,-neovariant2.x,-original_umis,-neovariant1_umis,-neovariant2_umis) %>%
                 dplyr:: rename("original"="original.y","neovariant1"="neovariant1.y","neovariant2"="neovariant2.y","original_umis"="original_umis.y","neovariant1_umis"="neovariant1_umis.y","neovariant2_umis"="neovariant2_umis.y")

In [ ]:
# Do not forget to annotate the invalid cell, the ones that have only one nucl (20 cells)
#subset_n_event %>% filter(rowSums(is.na(select(., original_umis.y, neovariant1_umis.y, neovariant2_umis.y))) > 1)

In [ ]:
single_events <- events_v31 %>% anti_join(multiple_events %>% 
                        select(subject,cell,position,original,neovariant1,neovariant2), by=c("subject","cell","position"))

In [ ]:
#270 + 969 = 1239 events
events_v4 <- single_events %>% bind_rows(multiple_events)

In [ ]:
head(events_v4)